<a href="https://colab.research.google.com/github/niranjansendilkumar11/Irish-weather-pipeline/blob/main/ca2_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import requests



In [ ]:
#API Configuration
from google.colab import userdata

try:
  API_KEY =userdata.get('OWM_API_KEY')
  print("API Key loaded successfully!")
except Exception:
  API_KEY = ""


API Key loaded successfully!


In [ ]:
IRISH_CITIES = [
    "Dublin,IE",
    "Cork,IE",
    "Galway,IE",
    "Limerick,IE",
    "Waterford,IE",
    "Swords,IE",
    "Sligo,IE",
    "Derry,IE",
    "Drogheda,IE",
    "Tallaght,IE"
]
print(f"Monitoring {len(IRISH_CITIES)} Irish cities:")
for city in IRISH_CITIES:
    print(f"{city.split(',')[0]}")


Monitoring 10 Irish cities:
Dublin
Cork
Galway
Limerick
Waterford
Swords
Sligo
Derry
Drogheda
Tallaght


In [ ]:
import time

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

def fetch_city_weather(city_name):
    params = {
        "q": city_name,
        "appid": API_KEY,
        "units": "metric"
    }
    try:
      response = requests.get(BASE_URL, params=params,timeout=10)
      if response.status_code == 200:
        data = response.json()
        return {
                "city":          data["name"],
                "country":       data["sys"]["country"],
                "latitude":      data["coord"]["lat"],
                "longitude":     data["coord"]["lon"],
                "temp_celsius":  data["main"]["temp"],
                "feels_like":    data["main"]["feels_like"],
                "temp_min":      data["main"]["temp_min"],
                "temp_max":      data["main"]["temp_max"],
                "humidity":      data["main"]["humidity"],
                "pressure":      data["main"]["pressure"],
                "visibility":    data.get("visibility", None),
                "wind_speed":    data["wind"]["speed"],
                "wind_degree":   data["wind"].get("deg", None),
                "weather_main":  data["weather"][0]["main"],
                "weather_desc":  data["weather"][0]["description"],
                "cloud_coverage":data["clouds"]["all"],
                "sunrise":       data["sys"]["sunrise"],
                "sunset":        data["sys"]["sunset"],
                "fetched_at":    datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            }
      else:
            print(f"Failed for {city_name} - Status: {response.status_code}")
            return None
    except requests.exceptions.Timeout:
        print(f"Timeout for {city_name}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error for {city_name}: {e}")
        return None


In [ ]:
result = fetch_city_weather ("Dublin,IE")
print(result)

{'city': 'Dublin', 'country': 'IE', 'latitude': 53.344, 'longitude': -6.2672, 'temp_celsius': 10.78, 'feels_like': 9.4, 'temp_min': 10.32, 'temp_max': 11.6, 'humidity': 57, 'pressure': 1021, 'visibility': 10000, 'wind_speed': 11.83, 'wind_degree': 280, 'weather_main': 'Clouds', 'weather_desc': 'broken clouds', 'cloud_coverage': 75, 'sunrise': 1774764365, 'sunset': 1774810405, 'fetched_at': '2026-03-29 18:50:46'}


In [ ]:
#Section 4 - Data Acquisition
# This section fetches live weather data for all 10 Irish cities
def fetch_all_cities(cities):
  all_records = []
  for city in cities:
    print(f"Fetching {city.split(',')[0]}...")
    record = fetch_city_weather(city)
    if record:
      all_records.append(record)
    time.sleep(1)
  df_raw = pd.DataFrame(all_records)
  print(f"Done Fetched {len(all_records)} cities")
  return df_raw

df_raw = fetch_all_cities(IRISH_CITIES)


Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetching Swords...
Fetching Sligo...
Fetching Derry...
Fetching Drogheda...
Fetching Tallaght...
Done Fetched 10 cities


In [ ]:
# Displaying the raw  data we fetched
df_raw.head(10)

,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,visibility,wind_speed,wind_degree,weather_main,weather_desc,cloud_coverage,sunrise,sunset,fetched_at
0,Dublin,IE,53.3440,-6.2672,10.78,9.40,10.32,11.60,57,1021,10000,11.83,280,Clouds,broken clouds,75,1774764365,1774810405,2026-03-29 18:50:46
1,Cork,IE,51.8980,-8.4706,11.87,10.52,11.51,11.87,54,1026,10000,5.66,280,Clouds,broken clouds,75,1774764959,1774810868,2026-03-29 18:50:47
2,Galway,IE,53.2719,-9.0489,9.36,6.02,9.36,9.36,77,1025,10000,7.35,282,Clouds,scattered clouds,29,1774765035,1774811070,2026-03-29 18:50:48
3,Limerick,IE,52.6647,-8.6231,9.97,7.14,9.97,9.97,82,1026,10000,6.19,289,Clouds,scattered clouds,43,1774764961,1774810939,2026-03-29 18:50:49
4,Waterford,IE,52.2583,-7.1119,11.10,9.39,11.10,11.40,43,1024,10000,5.14,310,Clear,clear sky,0,1774764618,1774810558,2026-03-29 18:50:51
5,Swords,IE,53.4597,-6.2181,10.38,8.96,10.26,11.54,57,1021,10000,11.83,280,Clouds,broken clouds,75,1774764348,1774810399,2026-03-29 18:50:52
6,Sligo,IE,54.2697,-8.4694,8.96,5.00,8.96,8.96,71,1022,7000,9.26,270,Clouds,broken clouds,75,1774764848,1774810979,2026-03-29 18:50:53
7,Derry,IE,51.5867,-9.0503,10.43,9.85,10.43,10.43,89,1027,10000,7.59,285,Clouds,overcast clouds,99,1774765112,1774810994,2026-03-29 18:50:54
8,Drogheda,IE,53.7189,-6.3478,9.76,6.21,9.76,9.76,65,1021,10000,8.62,289,Clouds,overcast clouds,100,1774764366,1774810443,2026-03-29 18:50:55
9,Tallaght,IE,53.2859,-6.3734,10.72,9.28,9.73,11.01,55,1021,10000,11.83,280,Clouds,few clouds,20,1774764393,1774810428,2026-03-29 18:50:56


In [ ]:
#section 5- Transformations
# This section cleans and enriches the raw weather data

def categorise_temperature (temp_celsius):
  if temp_celsius < 0:
    return "Freezing"
  elif temp_celsius < 8:
        return "Cold"
  elif temp_celsius < 14:
        return "Cool"
  elif temp_celsius < 19:
        return "Mild"
  else:
        return "Warm"

In [ ]:
def categorise_wind(wind_speed):
    if wind_speed < 3.0:
        return "Calm"
    elif wind_speed < 8.0:
        return "Breezy"
    elif wind_speed < 14.0:
        return "Windy"
    else:
        return "Storm"


In [ ]:
def categorise_humidity(humidity):
  if humidity < 40:
        return "Dry"
  elif humidity < 70:
        return "Comfortable"
  else:
        return "Humid"

In [ ]:
#calculating how many hours of daylight each city gets
def convert_unix_timestamp(unix_time):
    return datetime.fromtimestamp(unix_time).strftime('%H:%M:%S')

def calculate_daylight_hours(sunrise_unix, sunset_unix):
    duration_seconds = sunset_unix - sunrise_unix
    return round(duration_seconds / 3600, 2)

print(convert_unix_timestamp(1774333095))
print(calculate_daylight_hours(1774333095, 1774377864))

06:18:15
12.44


In [ ]:
# Applies all transformations to the raw dataframe and returns a cleaned enriched version
def transform_weather_data(df):
  df_transformed = df.copy()

  df_transformed["sunrise_time"] = df_transformed["sunrise"].apply(convert_unix_timestamp)
  df_transformed["sunset_time"] = df_transformed["sunset"].apply(convert_unix_timestamp)
  df_transformed["daylight_hours"] = df_transformed.apply(
        lambda row: calculate_daylight_hours(row["sunrise"], row["sunset"]), axis=1
    )
  df_transformed["temp_category"] = df_transformed["temp_celsius"].apply(categorise_temperature)
  df_transformed["wind_category"] = df_transformed["wind_speed"].apply(categorise_wind)
  df_transformed["humidity_category"] = df_transformed["humidity"].apply(categorise_humidity)
  df_transformed["visibility_km"] = (df_transformed["visibility"] / 1000).round(2)
  df_transformed.drop(columns=["sunrise", "sunset", "visibility"], inplace=True)

  return df_transformed

df_transformed = transform_weather_data(df_raw)
print("Transformations applied successfully!")
print(f"Columns now: {df_transformed.shape[1]}")
df_transformed.head(10)

Transformations applied successfully!
Columns now: 23


,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,...,weather_desc,cloud_coverage,fetched_at,sunrise_time,sunset_time,daylight_hours,temp_category,wind_category,humidity_category,visibility_km
0,Dublin,IE,53.3440,-6.2672,10.78,9.40,10.32,11.60,57,1021,...,broken clouds,75,2026-03-29 18:50:46,06:06:05,18:53:25,12.79,Cool,Windy,Comfortable,10.0
1,Cork,IE,51.8980,-8.4706,11.87,10.52,11.51,11.87,54,1026,...,broken clouds,75,2026-03-29 18:50:47,06:15:59,19:01:08,12.75,Cool,Breezy,Comfortable,10.0
2,Galway,IE,53.2719,-9.0489,9.36,6.02,9.36,9.36,77,1025,...,scattered clouds,29,2026-03-29 18:50:48,06:17:15,19:04:30,12.79,Cool,Breezy,Humid,10.0
3,Limerick,IE,52.6647,-8.6231,9.97,7.14,9.97,9.97,82,1026,...,scattered clouds,43,2026-03-29 18:50:49,06:16:01,19:02:19,12.77,Cool,Breezy,Humid,10.0
4,Waterford,IE,52.2583,-7.1119,11.10,9.39,11.10,11.40,43,1024,...,clear sky,0,2026-03-29 18:50:51,06:10:18,18:55:58,12.76,Cool,Breezy,Comfortable,10.0
5,Swords,IE,53.4597,-6.2181,10.38,8.96,10.26,11.54,57,1021,...,broken clouds,75,2026-03-29 18:50:52,06:05:48,18:53:19,12.79,Cool,Windy,Comfortable,10.0
6,Sligo,IE,54.2697,-8.4694,8.96,5.00,8.96,8.96,71,1022,...,broken clouds,75,2026-03-29 18:50:53,06:14:08,19:02:59,12.81,Cool,Windy,Humid,7.0
7,Derry,IE,51.5867,-9.0503,10.43,9.85,10.43,10.43,89,1027,...,overcast clouds,99,2026-03-29 18:50:54,06:18:32,19:03:14,12.74,Cool,Breezy,Humid,10.0
8,Drogheda,IE,53.7189,-6.3478,9.76,6.21,9.76,9.76,65,1021,...,overcast clouds,100,2026-03-29 18:50:55,06:06:06,18:54:03,12.80,Cool,Windy,Comfortable,10.0
9,Tallaght,IE,53.2859,-6.3734,10.72,9.28,9.73,11.01,55,1021,...,few clouds,20,2026-03-29 18:50:56,06:06:33,18:53:48,12.79,Cool,Windy,Comfortable,10.0


In [ ]:
# Section 6 - Unit Testing
#Testing each transformation function to verify they return correct results
import unittest

class TestWeatherData(unittest.TestCase):

  def test_categorise_temperature(self):
    self.assertEqual(categorise_temperature(-5), "Freezing")
    self.assertEqual(categorise_temperature(4), "Cold")
    self.assertEqual(categorise_temperature(10), "Cool")
    self.assertEqual(categorise_temperature(16), "Mild")
    self.assertEqual(categorise_temperature(22), "Warm")

  def test_categorise_wind(self):
    self.assertEqual(categorise_wind(1.5), "Calm")
    self.assertEqual(categorise_wind(5.0), "Breezy")
    self.assertEqual(categorise_wind(10.0), "Windy")
    self.assertEqual(categorise_wind(18.0), "Storm")

  def test_categorise_humidity(self):
    self.assertEqual(categorise_humidity(30), "Dry")
    self.assertEqual(categorise_humidity(55), "Comfortable")
    self.assertEqual(categorise_humidity(85), "Humid")

  def test_calculate_daylight_hours(self):
    self.assertEqual(calculate_daylight_hours(1774333095, 1774377864), 12.44)

unittest.main(argv=[""], exit=False, verbosity=2)

test_calculate_daylight_hours (__main__.TestWeatherData.test_calculate_daylight_hours) ... ok
test_categorise_humidity (__main__.TestWeatherData.test_categorise_humidity) ... ok
test_categorise_temperature (__main__.TestWeatherData.test_categorise_temperature) ... ok
test_categorise_wind (__main__.TestWeatherData.test_categorise_wind) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.008s

OK


In [ ]:
# Section 7 - Integration Test
# Testing the full pipeline from data fetching to transformation

def test_full_pipeline():
    print("Running integration test...")

    #Fetching data for only one city
    test_data = fetch_city_weather("Dublin,IE")
    assert test_data is not None, "Fetch failed - no data returned"
    assert "temp_celsius" in test_data, "temp_celsius missing from fetched data"
    assert "humidity" in test_data, "humidity missing from fetched data"
    print("Data fetched successfully")

    test_df = pd.DataFrame([test_data])
    test_df_transformed = transform_weather_data(test_df)
    assert "temp_category" in test_df_transformed.columns, "temp_category missing from transformed data"
    assert "wind_category" in test_df_transformed.columns, "wind_category column missing"
    assert "sunrise_time" in test_df_transformed.columns, "sunrise_time column missing"
    print("Transformations applied successfully")

    assert test_df_transformed.shape[0] == 1, "Expected 1 row"
    assert test_df_transformed.shape[1] == 23, "Expected 23 columns"

    print("Integration test passed")

test_full_pipeline()



Running integration test...
Data fetched successfully
Transformations applied successfully
Integration test passed


In [ ]:
#Section 8 - Save data to CSV locally every hour in a CSV file
'''
import os

def save_to_csv(df, filename="irish_weather_data.csv"):

  if os.path.exists(filename):
    df.to_csv(filename, mode='a', header=False, index=False)
    print(f"Data appended to {filename}")
  else:
     df.to_csv(filename, mode='w', header=True, index=False)
     print(f"New file created: {filename}")

  print(f"Rows saved: {len(df)}")

save_to_csv(df_transformed) '''




'\nimport os\n\ndef save_to_csv(df, filename="irish_weather_data.csv"):\n\n  if os.path.exists(filename):\n    df.to_csv(filename, mode=\'a\', header=False, index=False)\n    print(f"Data appended to {filename}")\n  else:\n     df.to_csv(filename, mode=\'w\', header=True, index=False)\n     print(f"New file created: {filename}")\n\n  print(f"Rows saved: {len(df)}")\n\nsave_to_csv(df_transformed) '

In [ ]:
# Section 9 - Hourly Scheduler

'''def run_pipeline():
    print(f"Pipeline started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    df_raw = fetch_all_cities(IRISH_CITIES)

    df_transformed = transform_weather_data(df_raw)

    save_to_csv(df_transformed)

    print(f"Pipeline completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total rows in file so far:")
    df_check = pd.read_csv("irish_weather_data.csv")
    print(df_check.shape)

run_pipeline()

#running every hour
while True:
    print("Waiting 1 hour before next run...")
    time.sleep(3600)
    run_pipeline()'''




'def run_pipeline():\n    print(f"Pipeline started at: {datetime.now().strftime(\'%Y-%m-%d %H:%M:%S\')}")\n\n    df_raw = fetch_all_cities(IRISH_CITIES)\n\n    df_transformed = transform_weather_data(df_raw)\n\n    save_to_csv(df_transformed)\n\n    print(f"Pipeline completed at: {datetime.now().strftime(\'%Y-%m-%d %H:%M:%S\')}")\n    print(f"Total rows in file so far:")\n    df_check = pd.read_csv("irish_weather_data.csv")\n    print(df_check.shape)\n\nrun_pipeline()\n\n#running every hour\nwhile True:\n    print("Waiting 1 hour before next run...")\n    time.sleep(3600)\n    run_pipeline()'

In [ ]:
# Section 10 - Load data to Google Cloud SQL
# Connecting to  PostgreSQL database on Google Cloud

from sqlalchemy import create_engine, text

DB_HOST = userdata.get('DB_HOST')
DB_NAME = userdata.get('DB_NAME')
DB_USER = userdata.get('DB_USER')
DB_PASS = userdata.get('DB_PASS')

engine = create_engine(
    "postgresql+psycopg2://",
    creator=lambda: __import__('psycopg2').connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        port=5432
    )
)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Connected to Cloud SQL successfully!")
except Exception as e:
    print(f"Connection failed: {e}")

Connected to Cloud SQL successfully!


In [ ]:
# Creates the weather table in Cloud SQL

def load_to_cloud_sql(df, engine):

   create_table_query = text("""
        CREATE TABLE IF NOT EXISTS weather_data (
            id SERIAL PRIMARY KEY,
            city VARCHAR(100),
            country VARCHAR(10),
            latitude FLOAT,
            longitude FLOAT,
            temp_celsius FLOAT,
            feels_like FLOAT,
            temp_min FLOAT,
            temp_max FLOAT,
            humidity INTEGER,
            pressure INTEGER,
            wind_speed FLOAT,
            wind_degree FLOAT,
            weather_main VARCHAR(100),
            weather_desc VARCHAR(100),
            cloud_coverage INTEGER,
            fetched_at VARCHAR(50),
            sunrise_time VARCHAR(20),
            sunset_time VARCHAR(20),
            daylight_hours FLOAT,
            temp_category VARCHAR(20),
            wind_category VARCHAR(20),
            humidity_category VARCHAR(20),
            visibility_km FLOAT
        )
    """)

   with engine.connect() as conn:
        conn.execute(create_table_query)
        conn.commit()
   print("Table created or already exists")

    # Load dataframe into the table
   df.to_sql(
        name="weather_data",
        con=engine,
        if_exists="append",
        index=False
    )
   print(f"Successfully loaded {len(df)} rows to Cloud SQL")

load_to_cloud_sql(df_transformed, engine)

Table created or already exists
Successfully loaded 10 rows to Cloud SQL


In [ ]:
# Verifing data was loaded correctly

with engine.connect() as conn:
    result = conn.execute(text("SELECT city, temp_celsius, temp_category, wind_category, humidity_category FROM weather_data"))
    rows = result.fetchall()
    print(f"Total rows in database: {len(rows)}")

    for row in rows:
      print(f"{row[0]} | {row[1]}C | {row[2]} | {row[3]} | {row[4]}")

Total rows in database: 20
Dublin | 4.24C | Cold | Breezy | Humid
Cork | 3.87C | Cold | Breezy | Humid
Galway | 4.8C | Cold | Breezy | Humid
Limerick | 3.97C | Cold | Breezy | Humid
Waterford | 3.99C | Cold | Calm | Comfortable
Swords | 3.44C | Cold | Breezy | Humid
Sligo | 4.37C | Cold | Windy | Humid
Derry | 4.51C | Cold | Breezy | Humid
Drogheda | 2.83C | Cold | Breezy | Humid
Tallaght | 3.64C | Cold | Breezy | Humid
Dublin | 10.78C | Cool | Windy | Comfortable
Cork | 11.87C | Cool | Breezy | Comfortable
Galway | 9.36C | Cool | Breezy | Humid
Limerick | 9.97C | Cool | Breezy | Humid
Waterford | 11.1C | Cool | Breezy | Comfortable
Swords | 10.38C | Cool | Windy | Comfortable
Sligo | 8.96C | Cool | Windy | Humid
Derry | 10.43C | Cool | Breezy | Humid
Drogheda | 9.76C | Cool | Windy | Comfortable
Tallaght | 10.72C | Cool | Windy | Comfortable


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM weather_data"))
    print(result.fetchone())

(20,)
